In [1]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)

('input.txt', <http.client.HTTPMessage at 0x7ae112b800e0>)

## Setup & Data Loading

Notes from KV cache implementation:
- During inference (after prefill), only one new token arrives each step
- Q is computed from only the new token (size 1)
- K and V can be concatenated onto whatever you already cached
- Q @ K^T still works if Q has shape (B, 1, hs) and K has shape (B, T, hs) → result is (B, 1, T)
- We don't need the causal mask during decode because we are only considering the most recent token

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss, _ = model(X, Y)  # unpack 3 return values now
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## Hint 1: Request Dataclass

Each in-flight generation carries its own state:
- `prompt_tokens` — the initial context
- `max_new_tokens` — individual stopping condition (request 0 may want 20, request 1 may want 100)
- `generated_tokens` — accumulates one token per scheduler step
- `status` — lets the scheduler know whether to batch this request
- `kv_cache` — **per-request** KV cache, keyed by `(layer_idx, head_idx)`

If request 0 finishes after 20 tokens but request 1 needs 100, request 0 is evicted from the batch (its row disappears), and request 1 continues generating.

In [ ]:
@dataclass
class Request:
    """Each in-flight generation carries its own state and KV cache."""
    id: int
    prompt_tokens: List[int]          # the original encoded prompt
    max_new_tokens: int               # how many tokens this request wants
    generated_tokens: List[int] = field(default_factory=list)
    status: str = "waiting"           # "waiting" -> "prefilling" -> "active" -> "done"
    prefill_cursor: int

    # Hint 2: Per-request KV cache, keyed by (layer_idx, head_idx)
    # Each value is a (key_tensor, value_tensor) tuple of shape (1, T_i, head_size)
    # T_i grows by 1 each decode step — different requests have different T_i
    kv_cache: Dict[Tuple[int, int], Tuple[torch.Tensor, torch.Tensor]] = field(
        default_factory=dict
    )

    @property
    def tokens_so_far(self) -> List[int]:
        """Full sequence: prompt + everything generated."""
        return self.prompt_tokens + self.generated_tokens

    @property
    def num_generated(self) -> int:
        return len(self.generated_tokens)

    @property
    def is_done(self) -> bool:
        return self.num_generated >= self.max_new_tokens
    
    @property
    def is_fully_prefilled(self) -> bool:
        return self.prefill_cursor == len(prompt_tokens)

    def clear_cache(self):
        self.kv_cache.clear()

## Hint 2: Stateless Head — KV Cache Moved Outside the Model

**Before:** `Head` owned `self.key_cache` and `self.value_cache` — one monolithic `(B, T, hs)` tensor.
This breaks when different requests have different sequence lengths.

**After:** `Head` is stateless. The cache is:
1. Passed **into** `forward()` as `past_k, past_v`
2. Returned **out of** `forward()` as updated `(new_k, new_v)`
3. **Stored on the `Request` object**, keyed by `(layer_idx, head_idx)`

This threads through: `Head` → `MultiHeadAttention` → `Block` → `GPTLanguageModel`.

Also changed `nn.Sequential` → `nn.ModuleList` for `self.blocks` so we can pass
per-block cache into each block individually.

In [4]:
class Head(nn.Module):
    """One head of self-attention — now STATELESS (no internal cache)."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        # NOTE: no key_cache / value_cache attributes anymore!

    def forward(self, x, past_k=None, past_v=None, attn_mask=None):
        """
        Args:
            x:      (B, T, C)       input embeddings
            past_k: (B, T_past, hs) cached keys, or None
            past_v: (B, T_past, hs) cached values, or None
        Returns:
            out:   (B, T, hs)           attention output
            new_k: (B, T_past+T, hs)    updated key cache   (None during training)
            new_v: (B, T_past+T, hs)    updated value cache  (None during training)
        """
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        v = self.value(x)  # (B, T, hs)

        if not self.training:
            if past_k is not None:
                # ── Decode step: append new K/V onto cached past ──
                k = torch.cat([past_k, k], dim=1)  # (B, T_past + T, hs)
                v = torch.cat([past_v, v], dim=1)

                # Q attends over full cache — no causal mask needed (T=1)
                wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5

                if attn_mask is not None:
                    new_valid = torch.ones(B, 1, T, device=wei.device, dtype=torch.bool)
                    full_mask = torch.cat([attn_mask, new_valid], dim=-1)
                    wei = wei.masked_fill(~full_mask, float('-inf'))

                wei = F.softmax(wei, dim=-1)
                wei = self.dropout(wei)
                out = wei @ v
            else:
                # ── Prefill step: full prompt, needs causal mask ──
                wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
                wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
                wei = F.softmax(wei, dim=-1)
                wei = self.dropout(wei)
                out = wei @ v

            return out, k, v   # return updated cache
        else:
            # ── Training path — unchanged, no cache ──
            wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            out = wei @ v
            return out, None, None


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Args:
            x:       (B, T, C)
            past_kv: list of (past_k, past_v) per head, or None
        Returns:
            out:    (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head
        """
        if past_kv is None:
            past_kv = [(None, None)] * len(self.heads)

        outputs, new_kvs = [], []
        for i, h in enumerate(self.heads):
            pk, pv = past_kv[i]
            out, nk, nv = h(x, pk, pv, attn_mask=attn_mask)
            outputs.append(out)
            new_kvs.append((nk, nv))

        out = torch.cat(outputs, dim=-1)
        out = self.dropout(self.proj(out))
        return out, new_kvs


class FeedFoward(nn.Module):
    """A simple linear layer followed by a non-linearity."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communication followed by computation."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Returns:
            x:      (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head in this block
        """
        sa_out, new_kv = self.sa(self.ln1(x), past_kv, attn_mask=attn_mask)
        x = x + sa_out
        x = x + self.ffwd(self.ln2(x))
        return x, new_kv


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # ModuleList instead of Sequential so we can pass per-block cache
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, pos=None, past_kvs=None, attn_mask=None):
        """
        Args:
            idx:      (B, T) token indices
            targets:  (B, T) target indices, or None
            pos:      (B, T) explicit position indices, or None (uses arange)
            past_kvs: list-of-lists cache structure, or None
                      past_kvs[layer][head] = (key_tensor, value_tensor)
        Returns:
            logits:   (B, T, vocab_size)
            loss:     scalar or None
            new_kvs:  updated cache with same structure as past_kvs
        """
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)

        if pos is None:
            pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        else:
            pos_emb = self.position_embedding_table(pos)  # (B, T, C)

        x = tok_emb + pos_emb  # (B, T, C)

        # Thread cache through each block
        if past_kvs is None:
            past_kvs = [None] * len(self.blocks)

        new_kvs = []
        for i, block in enumerate(self.blocks):
            x, block_kv = block(x, past_kvs[i], attn_mask=attn_mask)
            new_kvs.append(block_kv)

        x = self.ln_f(x)          # (B, T, C)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss, new_kvs

    def generate(self, idx, max_new_tokens):
        """Original generate (no cache, full recompute) for reference."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Training

Training is **unchanged** — during `model.train()`, every `Head` takes the training branch
and returns `(out, None, None)` for the cache. The cache is simply discarded via `_`.

In [5]:
model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss, _ = model(xb, yb)  # _ discards the cache during training
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Quick sanity check with the original no-cache generate
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.1959, val loss 4.1962
step 100: train loss 2.6229, val loss 2.6166
step 200: train loss 2.4555, val loss 2.4488
step 300: train loss 2.3810, val loss 2.3928
step 400: train loss 2.3202, val loss 2.3223
step 500: train loss 2.2364, val loss 2.2541
step 600: train loss 2.1812, val loss 2.2234
step 700: train loss 2.1326, val loss 2.1583
step 800: train loss 2.0932, val loss 2.1352
step 900: train loss 2.0499, val loss 2.0987
step 1000: train loss 2.0349, val loss 2.0819
step 1100: train loss 1.9994, val loss 2.0706
step 1200: train loss 1.9857, val loss 2.0726
step 1300: train loss 1.9638, val loss 2.0401
step 1400: train loss 1.9386, val loss 2.0296
step 1500: train loss 1.9028, val loss 1.9947
step 1600: train loss 1.8821, val loss 1.9914
step 1700: train loss 1.8785, val loss 1.9813
step 1800: train loss 1.8746, val loss 1.9878
step 1900: train loss 1.8451, val loss 1.9599
step 2000: train loss 1.8294, val loss 1.9525
step 2100: train loss 1.

## Generation Functions

Three generation modes to compare:

1. **`generate_no_cache`** — full recompute every step (baseline)
2. **`generate_with_cache`** — KV cache passed externally as tensors (fast, single-request)
3. **`generate_request`** — uses the `Request` dataclass to store per-request cache (foundation for continuous batching)

In [6]:
# ── 1. No KV cache (full recompute every step) ───────────────────────────────
def generate_no_cache(model, idx, max_new_tokens):
    model.train()  # disables KV cache path
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _, _ = model(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
    return idx


# ── 2. With KV cache — passed as raw tensors ─────────────────────────────────
def generate_with_cache(model, idx, max_new_tokens):
    """KV cache stored externally — threaded through forward() each step."""
    model.eval()
    with torch.no_grad():
        # Prefill: run the entire prompt, get initial cache
        logits, _, past_kvs = model(idx)

        for step in range(max_new_tokens):
            logits = logits[:, -1, :]           # (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)

            # Decode: only the new token + its position
            curr_pos = torch.tensor([[idx.shape[1] - 1]], device=device)  # (1, 1)
            logits, _, past_kvs = model(idx_next, pos=curr_pos, past_kvs=past_kvs)

    return idx


# ── 3. Per-Request generation (Hint 1 + 2 combined) ──────────────────────────
def generate_request(model, request: Request):
    """
    Generate for a single Request object.
    The KV cache lives on the Request, not inside the model.

    This is the building block for the continuous batching scheduler (Hint 3).
    Each request independently owns its cache, so different requests
    can have different sequence lengths and lifetimes.
    """
    model.eval()
    with torch.no_grad():
        # Convert prompt to tensor
        prompt = torch.tensor(
            [request.prompt_tokens], dtype=torch.long, device=device
        )  # (1, T_prompt)

        # ── Prefill ──
        logits, _, new_kvs = model(prompt)

        # Store the cache on the request object
        # new_kvs[layer_idx][head_idx] = (key_tensor, value_tensor)
        for layer_idx, block_kv in enumerate(new_kvs):
            for head_idx, (k, v) in enumerate(block_kv):
                request.kv_cache[(layer_idx, head_idx)] = (k, v)

        request.status = "active"

        # ── Decode loop ──
        while not request.is_done:
            logits = logits[:, -1, :]           # (1, vocab_size)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (1, 1)

            request.generated_tokens.append(idx_next.item())

            if request.is_done:
                break

            # Rebuild past_kvs from request's per-request cache
            past_kvs = []
            for layer_idx in range(n_layer):
                block_kv = []
                for head_idx in range(n_head):
                    block_kv.append(request.kv_cache[(layer_idx, head_idx)])
                past_kvs.append(block_kv)

            curr_pos = torch.tensor(
                [[len(request.tokens_so_far) - 1]], device=device
            )  # (1, 1)
            logits, _, new_kvs = model(idx_next, pos=curr_pos, past_kvs=past_kvs)

            # Update the request's cache with the new K/V
            for layer_idx, block_kv in enumerate(new_kvs):
                for head_idx, (k, v) in enumerate(block_kv):
                    request.kv_cache[(layer_idx, head_idx)] = (k, v)

        request.status = "done"

## Test: Per-Request Generation

Verify that the `Request`-based generation produces coherent output,
and that multiple independent requests with different prompts and lengths
all complete correctly.

In [7]:
# ── Single request test ──────────────────────────────────────────────────────
print("=" * 60)
print("Test 1: Single request with Request object")
print("=" * 60)

req = Request(
    id=0,
    prompt_tokens=[0],  # newline character
    max_new_tokens=31,
)
generate_request(model, req)

print(f"Status: {req.status}")
print(f"Generated {req.num_generated} tokens")
print(f"Cache entries: {len(req.kv_cache)} (expected {n_layer * n_head})")
print()
print(decode(req.tokens_so_far))

Test 1: Single request with Request object
Status: done
Generated 31 tokens
Cache entries: 16 (expected 16)


Bear in Warmen liking than this


In [9]:
# ── Multiple independent requests with different prompts/lengths ─────────────
print("=" * 60)
print("Test 2: Multiple independent requests")
print("=" * 60)

requests = [
    Request(id=0, prompt_tokens=encode("O Romeo, "),     max_new_tokens=23),
    Request(id=1, prompt_tokens=encode("To be or "),     max_new_tokens=23),
    Request(id=2, prompt_tokens=encode("KING HENRY:"),   max_new_tokens=21),
]

for req in requests:
    generate_request(model, req)
    print(f"\n--- Request {req.id} ({req.status}, {req.num_generated} tokens) ---")
    print(decode(req.tokens_so_far))

# Verify all requests finished and have the right cache shape
for req in requests:
    assert req.status == "done", f"Request {req.id} not done!"
    assert req.num_generated == req.max_new_tokens, (
        f"Request {req.id}: expected {req.max_new_tokens}, got {req.num_generated}"
    )
    # Each cache entry should have T = len(prompt) + num_generated
    sample_k, _ = req.kv_cache[(0, 0)]  # layer 0, head 0
    expected_T = len(req.prompt_tokens) + req.num_generated - 1
    assert sample_k.shape[1] == expected_T, (
        f"Request {req.id}: cache T={sample_k.shape[1]}, expected {expected_T}"
    )

print("\n✓ All requests completed with correct cache shapes!")

Test 2: Multiple independent requests

--- Request 0 (done, 23 tokens) ---
O Romeo, ak
And in on her evicks

--- Request 1 (done, 23 tokens) ---
To be or well most rive wees in 

--- Request 2 (done, 21 tokens) ---
KING HENRY:
A poor my hours
To k

✓ All requests completed with correct cache shapes!


## Benchmark: No Cache vs External Cache

In [10]:
N_TOKENS   = 10
N_RUNS     = 3
context    = torch.zeros((1, 1), dtype=torch.long, device=device)

# warm-up
_ = generate_no_cache(model, context.clone(), 10)
_ = generate_with_cache(model, context.clone(), 10)

# --- No KV cache ---
times_no_cache = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    generate_no_cache(model, context.clone(), N_TOKENS)
    if device == 'cuda':
        torch.cuda.synchronize()
    times_no_cache.append(time.perf_counter() - t0)

# --- With KV cache ---
times_cache = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    generate_with_cache(model, context.clone(), N_TOKENS)
    if device == 'cuda':
        torch.cuda.synchronize()
    times_cache.append(time.perf_counter() - t0)

# --- Per-request cache ---
times_request = []
for _ in range(N_RUNS):
    req = Request(id=0, prompt_tokens=[0], max_new_tokens=N_TOKENS)
    t0 = time.perf_counter()
    generate_request(model, req)
    if device == 'cuda':
        torch.cuda.synchronize()
    times_request.append(time.perf_counter() - t0)

avg_no_cache = sum(times_no_cache) / N_RUNS
avg_cache    = sum(times_cache)    / N_RUNS
avg_request  = sum(times_request)  / N_RUNS

print(f"Tokens generated  : {N_TOKENS}")
print(f"No KV cache       : {avg_no_cache:.3f}s  ({N_TOKENS/avg_no_cache:.1f} tok/s)")
print(f"With KV cache     : {avg_cache:.3f}s  ({N_TOKENS/avg_cache:.1f} tok/s)")
print(f"Per-request cache : {avg_request:.3f}s  ({N_TOKENS/avg_request:.1f} tok/s)")
print(f"Speedup (cache)   : {avg_no_cache/avg_cache:.2f}×")
print(f"Speedup (request) : {avg_no_cache/avg_request:.2f}×")

Tokens generated  : 10
No KV cache       : 0.066s  (151.9 tok/s)
With KV cache     : 0.065s  (154.9 tok/s)
Per-request cache : 0.064s  (155.9 tok/s)
Speedup (cache)   : 1.02×
Speedup (request) : 1.03×


## What's Next: Hint 3 — The Scheduler Loop

Now that each request owns its own KV cache, the next step is the **scheduler loop**:

```
while there are active requests OR the waiting queue is non-empty:
    1. Check the waiting queue — can any new requests join the batch?
    2. Build the input tensor from ALL active requests (each contributes 1 token)
    3. Forward pass → get logits for all active requests at once
    4. Sample next token for each request
    5. Check: did any request hit max_new_tokens? → remove it, emit result
    6. Go to 1
```

The key challenge will be **padding the KV caches** to a common T dimension
when batching multiple requests (since they have different sequence lengths).
You'll need to `torch.cat` along dim=0 after padding along dim=1.

After un-batching the results, scatter the updated caches back to each request.

In [ ]:
def assemble_batch_cache(requests):
    """
    Gather per-request KV caches into batched tensors.
    LEFT-pads shorter caches so new tokens always land at the right edge.

    Big problem: You have 3 active requests. Each owns its own KV cache. You need to feed them to the model as one
    batched tensor. But their caches have different lengths:

    Returns:
        past_kvs:    batched cache structure  [layer][head] = (B, T_max, hs)
        attn_mask:   (B, 1, T_max) bool — True = valid, False = padding
        pad_lengths: list of int — how many pad positions per request (for disassembly)
    """

    B = len(requests)
    lengths = [req.kv_cache[(0, 0)][0].shape[1] for req in requests]
    max_t = max(lengths)

    pad_lengths = [max_t - t for t in lengths] # pad lengths for every position in t

    attn_mask = torch.zeros(B, 1, max_t, device=device, dtype=torch.bool)

    for i, pad in enumerate(pad_lengths):
        attn_mask[i, 0, pad:] = True

    past_kvs = []

    for layer_idx in range(n_layer):
        block_kv = []

        for head_idx in range(n_head):
            keys, values = [], []

            for i, req in enumerate(requests):
                k, v = req.kv_cache[(layer_idx, head_idx)]
                if pad_lengths[i] > 0:
                    hs = k.shape[2]
                    pad = torch.zeros(1, pad_lengths[i], hs, device=device)
                    k = torch.cat([pad, k], dim=1)
                    v = torch.cat([pad, v], dim=1)

                keys.append(k)
                values.append(v)

            block_kv.append((torch.cat(keys, dim=0), torch.cat(values, dim=0)))

        past_kvs.append(block_kv)

    return past_kvs, attn_mask, pad_lengths

def disassemble_batch_cache(requests, new_kvs, pad_lengths):
    """
    Scatter batched KV cache back to per-request storage.
    After Head's torch.cat, each row is (T_max + 1) — strip the left-padding.
    """
    for layer_idx, block_kv in enumerate(new_kvs):
        for head_idx, (batched_k, batched_v) in enumerate(block_kv):
            for i, req in enumerate(requests):
                pad = pad_lengths[i]
                req.kv_cache[(layer_idx, head_idx)] = (
                    batched_k[i : i + 1, pad:, :],      # (1, T_i + 1, hs)
                    batched_v[i : i + 1, pad:, :],
                )

def continuous_batching_generate(model, request_queue, max_batch_size=4, token_budget=None):
    """
        Hint 1: Decode requests will be highest priority. 

    """

    model.eval()

    active_requests = []
    completed_requests = []
    prefilling_requests = []
    step = 0 # simulating clock in a real time server
    queue_idx = 0 # for efficiency reasons

    with torch.no_grad():
        while active_requests or queue_idx < len(request_queue):
            time_step, req = request_queue[queue_idx]

            remaining_budget = token_budget

            remaining_budget -= len(active_requests)

            prefilling_requests.append(active_requests[queue_idx])
            req.status = "prefilling"


            if not active_requests:
                step += 1
                continue

            B_active = len(active_requests)

            batch_tokens = torch.cat([req._last_token for req in active_requests])

            batch_positions = torch.tensor([[len(req.tokens_so_far) - 1] for req in active_requests], device=device)

            past_kvs, attn_mask, pad_lengths = assemble_batch_cache(active_requests)

            logits, _, new_kvs = model(
                batch_tokens,
                pos=batch_positions,
                past_kvs=past_kvs,
                attn_mask=attn_mask
            )

            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            disassemble_batch_cache(active_requests, new_kvs, pad_lengths)

            for i, req in enumerate(active_requests):
                req.generated_tokens.append(idx_next[i].item())
                req._last_token = idx_next[i : i + 1]

            still_active = []
            for req in active_requests:
                if req.is_done:
                    req.status = "done"
                    completed_requests.append(req)
                    print(f"  [step {step}] Completed request {req.id} "
                          f"({req.num_generated} tokens)")

                else:
                    still_active.append(req)

            active_requests = still_active

            step += 1

    return completed_requests


In [24]:
# Simulate 3 requests arriving at different times with different lengths
request_queue = [
    (0,  Request(id=0, prompt_tokens=encode("O Romeo, "),     max_new_tokens=17)),
    (0,  Request(id=1, prompt_tokens=encode("To be or "),     max_new_tokens=22)),
    (3,  Request(id=2, prompt_tokens=encode("KING HENRY:\n"), max_new_tokens=15)),
]

print("=" * 60)
print("Continuous Batching — Simulated Arrivals")
print("=" * 60)

completed = continuous_batching_generate(model, request_queue, max_batch_size=4)

# Print results
for req in sorted(completed, key=lambda r: r.id):
    print(f"\n{'─'*40}")
    print(f"Request {req.id}  |  {req.num_generated} tokens  |  status: {req.status}")
    print(f"{'─'*40}")
    print(decode(req.tokens_so_far))

# Verify correctness
for req in completed:
    k, _ = req.kv_cache[(0, 0)]
    expected_T = len(req.prompt_tokens) + req.num_generated - 1
    assert k.shape[1] == expected_T, f"Req {req.id}: cache T={k.shape[1]}, expected {expected_T}"
    assert req.status == "done"
    assert req.num_generated == req.max_new_tokens

print("\n✓ All requests completed with correct cache shapes!")


Continuous Batching — Simulated Arrivals
  [step 0] Admitted request 0 (prompt=9, max_new=17)
  [step 0] Admitted request 1 (prompt=9, max_new=22)
  [step 3] Admitted request 2 (prompt=12, max_new=15)
  [step 15] Completed request 0 (17 tokens)
  [step 16] Completed request 2 (15 tokens)
  [step 20] Completed request 1 (22 tokens)

────────────────────────────────────────
Request 0  |  17 tokens  |  status: done
────────────────────────────────────────
O Romeo, moor and see-spea

────────────────────────────────────────
Request 1  |  22 tokens  |  status: done
────────────────────────────────────────
To be or head he true, let not 

────────────────────────────────────────
Request 2  |  15 tokens  |  status: done
────────────────────────────────────────
KING HENRY:
She thout to He

✓ All requests completed with correct cache shapes!


In [ ]:
"""
step 0: Admit req 0 (prompt=10), req 1 (prompt=9)  — prefill each, batch decode begins
         Decode batch = [req0, req1] → ONE forward pass, B=2
step 1: Decode batch = [req0, req1] → B=2
step 2: Decode batch = [req0, req1] → B=2
step 3: Admit req 2 (prompt=12) — prefill, joins batch
         Decode batch = [req0, req1, req2] → B=3
...
step N: req 2 finishes (15 tokens) → evicted
         Decode batch = [req0, req1] → B=2
step M: req 0 finishes (20 tokens) → evicted
         Decode batch = [req1] → B=1
step K: req 1 finishes (40 tokens) → done

"""